# 👔 Pydantic 파이프라인 + 채용 이력서 자동 검토

```
Phase . Core 학습 — Pydantic + RunnableParallel 핵심 부품
Project.   이력서 → 정보 추출 + 적합도 + 답장 + 종합 판단
```

## Phase 0. 환경 설정

In [ ]:
#!uv add langchain langchain-openai python-dotenv pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 10.9 MB/s eta 0:00:00


In [2]:


from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Phase 1. PydanticOutputParser — 자연어를 객체로

5단계 패턴: 스키마 → 파서 → 프롬프트 → LLM → 체인

In [3]:
class Summary(BaseModel):
    title: str = Field(description="짧은 제목")
    sentiment: Literal["positive", "negative", "neutral"]
    key_points: list[str] = Field(description="핵심 2-3개")

parser = PydanticOutputParser(pydantic_object=Summary)

prompt = ChatPromptTemplate.from_messages([
    ("system", "텍스트를 분석합니다. 한국어로.\n\n{format_instructions}"),
    ("human", "{text}"),
]).partial(format_instructions=parser.get_format_instructions())

chain_summary = prompt | llm | parser

r = chain_summary.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r)

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']


## Phase 2. RunnableParallel — 여러 체인 결합

In [6]:
# 자연어 답장 체인 추가 (StrOutputParser)
chain_notice = (
    ChatPromptTemplate.from_messages([
        ("system", "팀원에게 알릴 한 줄 안내문을 작성하세요."),
        ("human", "{text}"),
    ]) | llm | StrOutputParser()
)

combined = RunnableParallel(summary=chain_summary, notice=chain_notice)

r = combined.invoke({"text": "신제품 출시를 6월에서 7월로 연기하기로 했습니다."})
print(r["summary"])
print(r["notice"])

title='신제품 출시 연기' sentiment='neutral' key_points=['신제품 출시 일정 변경', '출시일이 6월에서 7월로 연기됨']
신제품 출시가 6월에서 7월로 연기되었음을 알려드립니다.


## Phase 3. RunnableLambda — 함수도 체인에 (결과는 Pydantic으로)

In [7]:
class Decision(BaseModel):
    urgent: bool
    note: str

def route(x) -> Decision:
    s = x["summary"]
    return Decision(urgent=(s.sentiment == "negative"), note=s.title)

chain_routed = combined | RunnableLambda(route)

d = chain_routed.invoke({"text": "신제품 출시 연기"})
print(d)

urgent=True note='신제품 출시 연기'


## Phase 4. RunnablePassthrough — 원본 보존

In [8]:
full = RunnableParallel(
    summary=chain_summary,
    notice=chain_notice,
    original=RunnablePassthrough(),
)
r = full.invoke({"text": "신제품 출시 연기"})
print(r["original"])  # 입력 그대로 통과

{'text': '신제품 출시 연기'}


---
# 👔 Project. 채용 이력서 자동 검토 시스템

```
이력서 → 정보 추출 + 적합도 평가 + 답장 초안 (동시)
       → 종합 판단 (인터뷰 / 보류 / 거절)
       → DB 저장 레코드
```

**채용 포지션**: 시니어 데이터 분석가 (Python/SQL/5년+, 우대: 리딩 경험)

In [9]:
SAMPLE = """김민지. 서울대 통계학과 졸업, 카카오 데이터 분석가 7년차.
Python·SQL 주력, Tableau 대시보드 3개 운영. 지난 2년간 A/B 테스트 플랫폼 리드,
주니어 3명 멘토링. 희망 연봉 8,500만원."""

# 분석 체인 — 이력서 정보 추출
class Candidate(BaseModel):
    name: str
    years: int = Field(description="총 경력 연수")
    skills: list[str]
    leadership: bool = Field(description="팀 리딩/멘토링 경험 명시 여부")
    expected_salary: Optional[int] = Field(default=None, description="희망 연봉(만원)")

p1 = PydanticOutputParser(pydantic_object=Candidate)
chain_candidate = (
    ChatPromptTemplate.from_messages([
        ("system", "이력서에서 정보를 추출합니다.\n\n{format_instructions}"),
        ("human", "{resume}"),
    ]).partial(format_instructions=p1.get_format_instructions())
    | llm | p1
)
print(chain_candidate.invoke({"resume": SAMPLE}))

name='김민지' years=7 skills=['Python', 'SQL', 'Tableau'] leadership=True expected_salary=8500


In [10]:
# 적합도 평가 체인
class Fit(BaseModel):
    score: int = Field(ge=0, le=10, description="포지션 적합도 0-10")
    strengths: list[str] = Field(description="강점 2-3개")
    salary_fit: Literal["under", "in_range", "over", "unknown"]

p2 = PydanticOutputParser(pydantic_object=Fit)
chain_fit = (
    ChatPromptTemplate.from_messages([
        ("system", "포지션: 시니어 데이터 분석가 (Python/SQL/5년+, 연봉 7,000-9,000만원).\n"
                  "이력서 적합도를 평가하세요.\n\n{format_instructions}"),
        ("human", "{resume}"),
    ]).partial(format_instructions=p2.get_format_instructions())
    | llm | p2
)
print(chain_fit.invoke({"resume": SAMPLE}))

score=9 strengths=['7년 이상의 데이터 분석 경험', 'Python 및 SQL에 대한 강력한 전문성', 'A/B 테스트 플랫폼 리드 경험 및 멘토링 능력'] salary_fit='in_range'


In [11]:
# 답장 초안 체인 (자연어)
chain_reply = (
    ChatPromptTemplate.from_messages([
        ("system", "지원자에게 보낼 따뜻한 1차 회신을 3-4문장으로."),
        ("human", "{resume}"),
    ]) | llm | StrOutputParser()
)

# 3개 체인 동시 실행
review_v1 = RunnableParallel(
    candidate=chain_candidate,
    fit=chain_fit,
    reply=chain_reply,
)

r = review_v1.invoke({"resume": SAMPLE})
print(f"{r['candidate'].name} | {r['candidate'].years}년 | 적합도 {r['fit'].score}/10")

김민지 | 7년 | 적합도 9/10


In [12]:
# 종합 판단 — RunnableLambda + Pydantic
class HiringDecision(BaseModel):
    action: Literal["interview", "hold", "reject"]
    reason: str
    reply: str

def decide(x) -> HiringDecision:
    score = x["fit"].score
    if score >= 8:
        return HiringDecision(action="interview", reason=f"적합도 {score}점", reply=x["reply"])
    elif score >= 5:
        return HiringDecision(action="hold", reason=f"적합도 {score}점, 2차 검토 필요", reply=x["reply"])
    return HiringDecision(action="reject", reason=f"적합도 {score}점, 요건 미충족", reply=x["reply"])

review_v2 = review_v1 | RunnableLambda(decide)
d = review_v2.invoke({"resume": SAMPLE})
print(f"🎯 {d.action} — {d.reason}")

🎯 interview — 적합도 9점


In [13]:
# 여러 이력서 batch — 실무에선 매주 수십 건
RESUMES = [
    {"resume": SAMPLE},
    {"resume": "박철수, 신입, 통계학과. SQL/Python 기초."},
    {"resume": "이영희, 10년차 분석가, 토스 결제 데이터 리드, A/B 100건+, 희망 9,000만원."},
]
for d in review_v2.batch(RESUMES):
    print(f"[{d.action:9s}] {d.reason}")

[interview] 적합도 9점
[reject   ] 적합도 2점, 요건 미충족
[interview] 적합도 9점


---
## 🎓 마무리

3개 부품(`PydanticOutputParser` + `RunnableParallel` + `RunnableLambda`)으로 채용 1차 스크리닝 시스템 완성.

**확장**: 면접 질문 체인 추가 → `RunnableParallel`에 한 줄만 더하면 됨.